# Reading the alpha-search landscape

This notebook is a short tour of the saved research artifacts. It focuses on the analytical decisions and what the results mean; the production logic remains in the tested `src/` modules.

**Research question:** when automated search finds an attractive cross-sectional equity signal, does the result occupy a stable region and survive honest holdout validation, or is it mainly the best outcome from many noisy trials?

**Headline decision:** the selected signal retained a 0.553 gross Sharpe on the test window but only 0.058 after a 5 bps/unit-turnover cost assumption. Its 95% moving-block bootstrap interval included weak outcomes. I would not claim a validated trading edge.

## 1. Load the complete search record

Every evaluated expression is retained. The combined artifact contains 500 random-search and 200 Bayesian-search evaluations; the holdout artifact contains only the validation-selected winner from each optimizer.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.landscape import run_clustering, run_pca, run_umap  # noqa: E402
from src.trajectory import build_feature_matrix  # noqa: E402

trajectory_path = project_root / "results" / "trajectory_combined.parquet"
if not trajectory_path.exists():
    trajectory_path = project_root / "results" / "trajectory_random.parquet"
trajectory = pd.read_parquet(trajectory_path)
holdout = pd.read_parquet(project_root / "results" / "out_of_sample_results.parquet")

print(f"Loaded {len(trajectory):,} logged evaluations from {trajectory_path.name}.")

## 2. Separate search performance from decision evidence

The unprefixed metrics in the trajectory are validation-window results. They are useful for comparing candidates, but they are not final evidence. The test rows below were produced only after one winner per optimizer was fixed.

In [ ]:
search_summary = trajectory.groupby("optimizer")["validation_sharpe"].agg(
    evaluations="count", mean="mean", median="median", best="max"
)
display(search_summary.style.format("{:.3f}", subset=["mean", "median", "best"]))

holdout_columns = [
    "optimizer",
    "expression",
    "selection_sharpe",
    "test_sharpe",
    "test_net_sharpe",
    "test_sharpe_ci_lower",
    "test_sharpe_ci_upper",
    "test_probabilistic_sharpe_ratio",
]
display(holdout[holdout_columns].style.format(precision=3))

Both optimizers selected the same unnormalized 120-day volume-ratio expression. The duplicate test metrics are therefore one candidate reached by two search procedures—not two independent confirmations. This distinction matters when interpreting convergence.

## 3. Map candidate definitions and outcomes

Each candidate becomes a standardized feature vector containing its primitive, lookback parameters, normalization choice, and backtest diagnostics. PCA gives a stable linear projection; UMAP emphasizes local neighborhoods; KMeans supplies an operational partition.

These axes are synthetic coordinates. Their absolute values are meaningless—the useful information is relative distance, grouping, and coverage.

In [ ]:
features = build_feature_matrix(trajectory)
pca_points = run_pca(features)
umap_points = run_umap(features)
clusters = run_clustering(features)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
pca_plot = axes[0].scatter(
    pca_points[:, 0], pca_points[:, 1], c=trajectory["validation_sharpe"], cmap="viridis"
)
axes[0].set(title="PCA: validation Sharpe", xlabel="PC 1", ylabel="PC 2")
fig.colorbar(pca_plot, ax=axes[0], label="Validation Sharpe")

umap_plot = axes[1].scatter(
    umap_points[:, 0], umap_points[:, 1], c=clusters["kmeans"], cmap="tab10"
)
axes[1].set(title="UMAP: candidate regions", xlabel="UMAP 1", ylabel="UMAP 2")
fig.colorbar(umap_plot, ax=axes[1], label="KMeans cluster")
plt.show()

## 4. Interpretation

The top validation decile is more concentrated than the candidate population as a whole. Repeating the analysis after removing every performance metric still places top candidates across 3 of 5 expression-only clusters, so the pattern is not purely created by embedding Sharpe beside the expression fields.

That result is descriptive, not a claim of local optima. The decisive evidence is the holdout audit: the gross result is cost-sensitive, the dependence-aware interval is wide, and the trial-adjusted probabilities remain below 50%. The appropriate research action is to reject the trading claim while retaining the volume-ratio hypothesis as a possible subject for better-controlled follow-up work.